# Notebook 1 — YOLOv8n Baseline (5-Fold Cross Validation)

**Project:** Lightweight Fall Detection for Elderly Care Using GhostNeck-YOLOv8 on Edge Devices  
**Dataset:** University of Strathclyde Fall Detection (474 original images, 2 classes)  
**Validation:** 5-Fold Stratified Cross Validation  
**Environment:** Google Colab (Tesla T4 GPU)

Each fold uses ~80% for training and ~20% for validation.  
Final metrics are reported as **mean ± std** across all 5 folds.

---
## Step 1 — Environment Setup

In [ ]:
!pip install ultralytics scikit-learn -q

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.0 MB/s eta 0:00:00
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


---
## Step 2 — Extract Dataset and Merge All Splits into One Pool

In [ ]:
import os, zipfile, shutil, yaml
from pathlib import Path
from collections import Counter

# ============================================================
# CONFIG
# ============================================================
DRIVE_ZIP   = "/content/drive/MyDrive/fall_detection_clean.zip"
EXTRACT_DIR = "/content/dataset_raw"
POOL_DIR    = "/content/dataset_pool"    # merged images + labels
OUTPUT_DIR  = "/content/runs"
K_FOLDS     = 5
SEED        = 42
# ============================================================

# Locate and extract zip
if not os.path.exists(DRIVE_ZIP):
    alt = DRIVE_ZIP.replace(".zip", "")
    if os.path.exists(alt):
        DRIVE_ZIP = alt
    else:
        raise FileNotFoundError(f"Dataset not found: {DRIVE_ZIP}")

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print(f"Extracted to {EXTRACT_DIR}")

# Find the actual dataset root (handle nested folders)
yaml_files = list(Path(EXTRACT_DIR).rglob("data.yaml"))
if not yaml_files:
    raise FileNotFoundError("data.yaml not found in extracted dataset.")
dataset_root = yaml_files[0].parent
print(f"Dataset root: {dataset_root}")

# Merge all splits (train/valid/test) into one pool
pool_img = Path(POOL_DIR) / "images"
pool_lbl = Path(POOL_DIR) / "labels"
if os.path.exists(POOL_DIR):
    shutil.rmtree(POOL_DIR)
pool_img.mkdir(parents=True)
pool_lbl.mkdir(parents=True)

img_count = 0
for split in ["train", "valid", "test", "val"]:
    img_dir = dataset_root / split / "images"
    lbl_dir = dataset_root / split / "labels"
    if not img_dir.exists():
        continue
    for img_path in img_dir.glob("*"):
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            shutil.copy2(str(img_path), str(pool_img / img_path.name))
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                shutil.copy2(str(lbl_path), str(pool_lbl / lbl_path.name))
            img_count += 1

print(f"\nMerged pool: {img_count} images")

# Count class distribution
cls_counts = Counter()
for lbl in pool_lbl.glob("*.txt"):
    for line in lbl.read_text().strip().splitlines():
        if line.strip():
            cls_counts[int(line.split()[0])] += 1

names = {0: "fall_detected", 1: "non_fall"}
print("Class distribution:")
for cls, count in sorted(cls_counts.items()):
    # Map class 1 and 2 to non_fall for display
    display_name = names.get(cls, f"class_{cls}")
    if cls in [1, 2]:
        display_name = "non_fall"
    print(f"  {display_name} (class {cls}): {count} boxes")

Extracted to /content/dataset_raw
Dataset root: /content/dataset_raw/fall_detection_clean

Merged pool: 474 images
Class distribution:
  fall_detected (class 0): 283 boxes
  non_fall (class 1): 134 boxes
  non_fall (class 2): 134 boxes


---
## Step 3 — Generate 5-Fold Stratified Splits

Each image is assigned a primary class (from its first annotation line).  
StratifiedKFold ensures each fold has the same class ratio.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

# Assign primary class to each image
all_images = sorted(list(pool_img.glob("*.jpg")) + list(pool_img.glob("*.png")))
image_names = []
image_classes = []

for img_path in all_images:
    lbl_path = pool_lbl / (img_path.stem + ".txt")
    if lbl_path.exists():
        text = lbl_path.read_text().strip()
        if text:
            primary_cls = int(text.splitlines()[0].split()[0])
            # Merge class 1 and 2 into class 1 for stratification
            stratify_cls = 0 if primary_cls == 0 else 1
            image_names.append(img_path.name)
            image_classes.append(stratify_cls)

image_names = np.array(image_names)
image_classes = np.array(image_classes)

print(f"Total images for K-Fold: {len(image_names)}")
print(f"  fall_detected: {np.sum(image_classes == 0)}")
print(f"  non_fall:      {np.sum(image_classes == 1)}")

# Generate fold indices
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(image_names, image_classes))

# Print fold distribution
print(f"\n{K_FOLDS}-Fold split:")
for i, (train_idx, val_idx) in enumerate(folds):
    train_cls = Counter(image_classes[train_idx])
    val_cls   = Counter(image_classes[val_idx])
    print(f"  Fold {i+1}: train={len(train_idx)} (fall={train_cls[0]}, non_fall={train_cls[1]})  |  "
          f"val={len(val_idx)} (fall={val_cls[0]}, non_fall={val_cls[1]})")

Total images for K-Fold: 474
  fall_detected: 277
  non_fall:      197

5-Fold split:
  Fold 1: train=379 (fall=221, non_fall=158)  |  val=95 (fall=56, non_fall=39)
  Fold 2: train=379 (fall=221, non_fall=158)  |  val=95 (fall=56, non_fall=39)
  Fold 3: train=379 (fall=222, non_fall=157)  |  val=95 (fall=55, non_fall=40)
  Fold 4: train=379 (fall=222, non_fall=157)  |  val=95 (fall=55, non_fall=40)
  Fold 5: train=380 (fall=222, non_fall=158)  |  val=94 (fall=55, non_fall=39)


---
## Step 4 — Train Baseline YOLOv8n on Each Fold

Training config is identical across all folds for reproducibility.  
Each fold saves its best weights and metrics separately.

In [ ]:
from ultralytics import YOLO

# ============================================================
# TRAINING CONFIG — same for every fold
# ============================================================
EPOCHS     = 100
BATCH_SIZE = 16
IMG_SIZE   = 640
# ============================================================

all_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    fold_num = fold_idx + 1
    print("\n" + "=" * 65)
    print(f"  FOLD {fold_num}/{K_FOLDS}  —  train: {len(train_idx)}  |  val: {len(val_idx)}")
    print("=" * 65)

    # --- Create fold-specific dataset directory ---
    fold_dir = Path(f"/content/fold_{fold_num}")
    if fold_dir.exists():
        shutil.rmtree(fold_dir)

    for split, indices in [("train", train_idx), ("val", val_idx)]:
        (fold_dir / split / "images").mkdir(parents=True)
        (fold_dir / split / "labels").mkdir(parents=True)
        for idx in indices:
            name = image_names[idx]
            stem = Path(name).stem
            # Copy image
            shutil.copy2(str(pool_img / name),
                         str(fold_dir / split / "images" / name))
            # Copy label
            lbl_name = stem + ".txt"
            if (pool_lbl / lbl_name).exists():
                shutil.copy2(str(pool_lbl / lbl_name),
                             str(fold_dir / split / "labels" / lbl_name))

    # --- Write fold-specific data.yaml ---
    fold_yaml = {
        "path":  str(fold_dir),
        "train": "train/images",
        "val":   "val/images",
        "nc":    3,
        "names": ["fall_detected", "sitting", "walking"],
    }
    fold_yaml_path = str(fold_dir / "data.yaml")
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_yaml, f, default_flow_style=False)

    # --- Train ---
    model = YOLO("yolov8n.pt")
    results = model.train(
        data    = fold_yaml_path,
        epochs  = EPOCHS,
        imgsz   = IMG_SIZE,
        batch   = BATCH_SIZE,
        project = OUTPUT_DIR,
        name    = f"baseline_fold{fold_num}",
        optimizer      = "SGD",
        lr0            = 0.01,
        lrf            = 0.01,
        momentum       = 0.937,
        weight_decay   = 0.0005,
        warmup_epochs  = 3.0,
        warmup_momentum= 0.8,
        mosaic       = 1.0,
        mixup        = 0.1,
        close_mosaic = 10,
        fliplr       = 0.5,
        flipud       = 0.0,
        degrees      = 5.0,
        translate    = 0.1,
        scale        = 0.5,
        shear        = 2.0,
        perspective  = 0.0,
        hsv_h        = 0.015,
        hsv_s        = 0.7,
        hsv_v        = 0.4,
        erasing      = 0.0,
        copy_paste   = 0.0,
        patience = 20,
        seed     = SEED,
        workers  = 2,
        exist_ok = True,
        verbose  = True,
    )

    # --- Evaluate ---
    best_pt = Path(OUTPUT_DIR) / f"baseline_fold{fold_num}" / "weights" / "best.pt"
    eval_model = YOLO(str(best_pt))
    metrics = eval_model.val(data=fold_yaml_path)

    model_size = best_pt.stat().st_size / (1024**2)
    param_count = sum(p.numel() for p in eval_model.model.parameters()) / 1e6

    # Get fall_detected (class 0) specific AP
    fall_ap50 = float(metrics.box.ap50[0]) if len(metrics.box.ap50) > 0 else 0.0
    fall_ap   = float(metrics.box.ap[0])   if len(metrics.box.ap) > 0   else 0.0

    fold_result = {
        "fold":       fold_num,
        "map50":      metrics.box.map50,
        "map50_95":   metrics.box.map,
        "precision":  metrics.box.mp,
        "recall":     metrics.box.mr,
        "fall_ap50":  fall_ap50,
        "fall_ap":    fall_ap,
        "params_m":   param_count,
        "size_mb":    model_size,
    }
    all_fold_metrics.append(fold_result)

    print(f"\n  Fold {fold_num} results:")
    print(f"    mAP@0.5        : {metrics.box.map50:.4f}")
    print(f"    mAP@0.5:0.95   : {metrics.box.map:.4f}")
    print(f"    Precision      : {metrics.box.mp:.4f}")
    print(f"    Recall         : {metrics.box.mr:.4f}")
    print(f"    fall_detected AP@0.5 : {fall_ap50:.4f}")

    # Clean up fold directory to save disk space
    shutil.rmtree(fold_dir)

print("\n" + "=" * 65)
print("All folds complete.")
print("=" * 65)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

  FOLD 1/5  —  train: 379  |  val: 95
Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_1/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, h

---
## Step 5 — Results Summary (Mean ± Std)

In [ ]:
import statistics

print("\n" + "=" * 70)
print("Baseline YOLOv8n — 5-Fold Cross Validation Results")
print("=" * 70)

# Per-fold table
print(f"\n{'Fold':<6} {'mAP@0.5':>10} {'mAP@50:95':>12} {'Precision':>12} {'Recall':>10} {'fall AP@0.5':>13}")
print("-" * 65)
for r in all_fold_metrics:
    print(f"  {r['fold']:<4} {r['map50']:>10.4f} {r['map50_95']:>12.4f} {r['precision']:>12.4f} {r['recall']:>10.4f} {r['fall_ap50']:>13.4f}")

# Mean ± Std
print("-" * 65)
metrics_keys = {
    'mAP@0.5':        'map50',
    'mAP@0.5:0.95':   'map50_95',
    'Precision':       'precision',
    'Recall':          'recall',
    'fall AP@0.5':     'fall_ap50',
    'fall AP@0.5:0.95':'fall_ap',
}

summary = {}
print(f"\n{'Metric':<20} {'Mean':>10} {'± Std':>10}")
print("-" * 42)
for display_name, key in metrics_keys.items():
    values = [r[key] for r in all_fold_metrics]
    mean = statistics.mean(values)
    std  = statistics.stdev(values) if len(values) > 1 else 0.0
    summary[key] = {'mean': mean, 'std': std}
    print(f"  {display_name:<18} {mean:>10.4f} {'±':>3} {std:<8.4f}")

# Model info (same across folds)
print(f"\n  Parameters : {all_fold_metrics[0]['params_m']:.2f}M")
print(f"  Model size : {all_fold_metrics[0]['size_mb']:.2f} MB")


Baseline YOLOv8n — 5-Fold Cross Validation Results

Fold      mAP@0.5    mAP@50:95    Precision     Recall   fall AP@0.5
-----------------------------------------------------------------
  1        0.8987       0.5894       0.9415     0.7917        0.9830
  2        0.8814       0.6022       0.8676     0.8360        0.9802
  3        0.8885       0.5911       0.9290     0.8173        0.9602
  4        0.7403       0.4901       0.7982     0.6925        0.9709
  5        0.9037       0.5950       0.8068     0.8364        0.9176
-----------------------------------------------------------------

Metric                     Mean      ± Std
------------------------------------------
  mAP@0.5                0.8625   ± 0.0688  
  mAP@0.5:0.95           0.5736   ± 0.0469  
  Precision              0.8686   ± 0.0666  
  Recall                 0.7948   ± 0.0600  
  fall AP@0.5            0.9624   ± 0.0266  
  fall AP@0.5:0.95       0.6751   ± 0.0557  

  Parameters : 3.01M
  Model size : 5.97 MB

---
## Step 6 — Save Results to Google Drive

In [ ]:
import json

SAVE_DIR = Path("/content/drive/MyDrive/fall_detection_kfold_results/baseline")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save per-fold metrics
results_dict = {
    "model":       "YOLOv8n Baseline",
    "k_folds":     K_FOLDS,
    "seed":        SEED,
    "epochs":      EPOCHS,
    "total_images": len(image_names),
    "per_fold":    all_fold_metrics,
    "summary":     {k: {"mean": round(v["mean"], 4), "std": round(v["std"], 4)}
                    for k, v in summary.items()},
}

with open(SAVE_DIR / "baseline_kfold_results.json", 'w') as f:
    json.dump(results_dict, f, indent=2)

# Save the best fold's weights (highest fall AP@0.5)
best_fold = max(all_fold_metrics, key=lambda x: x['fall_ap50'])
best_fold_pt = Path(OUTPUT_DIR) / f"baseline_fold{best_fold['fold']}" / "weights" / "best.pt"
if best_fold_pt.exists():
    shutil.copy2(str(best_fold_pt), str(SAVE_DIR / "best.pt"))
    print(f"Best fold: Fold {best_fold['fold']} (fall AP@0.5 = {best_fold['fall_ap50']:.4f})")
    print(f"Weights saved to: {SAVE_DIR / 'best.pt'}")

print(f"Results saved to: {SAVE_DIR}")
print(json.dumps(results_dict['summary'], indent=2))

Best fold: Fold 1 (fall AP@0.5 = 0.9830)
Weights saved to: /content/drive/MyDrive/fall_detection_kfold_results/baseline/best.pt
Results saved to: /content/drive/MyDrive/fall_detection_kfold_results/baseline
{
  "map50": {
    "mean": 0.8625,
    "std": 0.0688
  },
  "map50_95": {
    "mean": 0.5736,
    "std": 0.0469
  },
  "precision": {
    "mean": 0.8686,
    "std": 0.0666
  },
  "recall": {
    "mean": 0.7948,
    "std": 0.06
  },
  "fall_ap50": {
    "mean": 0.9624,
    "std": 0.0266
  },
  "fall_ap": {
    "mean": 0.6751,
    "std": 0.0557
  }
}
